# CSPF — اجرای یک‌کلیکی (NLI + آموزش consistency + κ)
فقط **یک سلول** را اجرا کن (Runtime ← T4 GPU). همه‌ی نتایج روی Google Drive در `BASE/results_FIXED/` ذخیره می‌شوند و بعد از قطع شدن، اجرای دوباره از همان‌جا ادامه می‌دهد.
در پایان یک فایل `CSPF_new_results_bundle.zip` ساخته می‌شود؛ همان را برای من بفرست.
برای κ، فایل پرشده‌ی ارزیاب دوم را با نام `annotation_sheet_for_second_annotator_FILLED.csv` در `BASE` بگذار و سلول را دوباره اجرا کن.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CSPF — ONE-CLICK UPGRADE RUN  (NLI fidelity → consistency training → κ if annotator-2 file exists)
# Everything is saved to Google Drive. Re-running after a disconnect resumes where it stopped.
# ═══════════════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/postdoc-papers/paper2"      # <- the only path you may need to edit

RUN_NLI, RUN_CONSISTENCY, RUN_KAPPA = True, True, True
import subprocess, sys, os, gc, time, glob, zipfile, traceback
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "sentencepiece"], check=False)   # does NOT touch numpy
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -> Runtime > Change runtime type > T4 GPU")
assert os.path.exists(f"{BASE}/ALL_MODELS_combined_FIXED.csv"), "ALL_MODELS_combined_FIXED.csv not found in BASE"
assert os.path.exists(f"{BASE}/annotated_output.csv"), "put annotated_output.csv (your 576 labelled pairs) in BASE"
RES = f"{BASE}/results_FIXED"; os.makedirs(RES, exist_ok=True)

STAGES = {}
STAGES["nli"] = r'''
FIXED_CSV = f"{BASE}/ALL_MODELS_combined_FIXED.csv"
HUMAN_CSV = f"{BASE}/annotated_output.csv"
OUT_DIR   = f"{BASE}/results_FIXED/nli"
NLI_MODEL = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
FULL_RUN  = False          # True = all pairs
PER_STRATUM = 1500         # sample size per (family x severity) stratum when FULL_RUN=False
CHUNK = 20000; BATCH = 128; MAXLEN = 160; SEED = 0
import os; os.makedirs(OUT_DIR+"/chunks", exist_ok=True)


import pandas as pd, numpy as np, os, glob, warnings; warnings.filterwarnings("ignore")
hdr = pd.read_csv(FIXED_CSV, nrows=0).columns.tolist()
MODELS = sorted({c[:-len("_pert_label")] for c in hdr if c.endswith("_pert_label")})
print(len(MODELS), "models")
meta = ["source_dataset","review_idx","original_review","perturbed_review","perturbation_family","severity","sem_preservation_score"]
use = meta + [f"{m}_{s}_label" for m in MODELS for s in ("orig","pert")]
parts=[]
for ch in pd.read_csv(FIXED_CSV, usecols=use, chunksize=100000, low_memory=False):
    fl = np.zeros(len(ch), dtype=np.float32)
    for m in MODELS: fl += (ch[f"{m}_orig_label"].values != ch[f"{m}_pert_label"].values)
    ch = ch[meta].copy(); ch["mean_flip"] = fl/len(MODELS); parts.append(ch)
df = pd.concat(parts, ignore_index=True); del parts
df["key"] = df.source_dataset.astype(str)+"::"+df.review_idx.astype(str)
df["pair_id"] = np.arange(len(df))
df["stratum"] = df.perturbation_family+"|"+df.severity.astype(str)
N_str = df.groupby("stratum").size()
if FULL_RUN: S = df.copy()
else:
    sh = df.sample(frac=1, random_state=SEED); S = sh[sh.groupby("stratum").cumcount() < PER_STRATUM].copy()
n_s = S.groupby("stratum").size(); S["w"] = S.stratum.map(N_str/n_s)       # inverse-probability weight -> corpus-level estimates
S = S.reset_index(drop=True); print(len(df), "pairs in corpus;", len(S), "scored")


import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
dev = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(NLI_MODEL); mdl = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(dev).eval()
if dev=="cuda": mdl.half()
l2i = {v.lower():k for k,v in mdl.config.id2label.items()}; E_, N_, C_ = l2i["entailment"], l2i["neutral"], l2i["contradiction"]
print("label map:", mdl.config.id2label)

@torch.no_grad()
def nli_probs(prem, hyp):
    """returns array [n,3] = P(entail), P(neutral), P(contradiction); length-sorted batches"""
    order = np.argsort([len(a)+len(b) for a,b in zip(prem,hyp)]); out = np.zeros((len(prem),3), dtype=np.float32)
    for i in range(0, len(order), BATCH):
        ix = order[i:i+BATCH]
        enc = tok([prem[j] for j in ix],[hyp[j] for j in ix], padding=True, truncation=True, max_length=MAXLEN, return_tensors="pt").to(dev)
        p = torch.softmax(mdl(**enc).logits.float(), -1).cpu().numpy()
        out[ix] = p[:,[E_,N_,C_]]
    return out

def score(frame, tag):
    """resumable: one .npy per chunk on Drive"""
    res=[]
    for s in range(0, len(frame), CHUNK):
        f = f"{OUT_DIR}/chunks/{tag}_{s}.npy"
        if os.path.exists(f): res.append(np.load(f)); continue
        sub = frame.iloc[s:s+CHUNK]; o=sub.original_review.astype(str).tolist(); p=sub.perturbed_review.astype(str).tolist()
        a = nli_probs(o,p); b = nli_probs(p,o); arr = np.hstack([a,b]); np.save(f, arr); res.append(arr); print(tag, s, "done")
    return np.vstack(res)


R = score(S, "main")
S[["ent_o2p","neu_o2p","con_o2p","ent_p2o","neu_p2o","con_p2o"]] = R
S["nli_bi"]  = np.minimum(S.ent_o2p, S.ent_p2o)
S["nli_con"] = np.maximum(S.con_o2p, S.con_p2o)
S[["pair_id","key","perturbation_family","severity","mean_flip","sem_preservation_score","w","ent_o2p","ent_p2o","nli_bi","nli_con"]].to_csv(f"{OUT_DIR}/nli_scores.csv", index=False)
print(S[["ent_o2p","ent_p2o","nli_bi","nli_con"]].describe().round(3))


# (a) per-family fidelity profile (weighted to corpus level)  (b) flip rate restricted to NLI-equivalent pairs
def wmean(x, w): return float(np.sum(x*w)/np.sum(w))
def prof(g):
    w=g.w.values
    return pd.Series(dict(n_scored=len(g), mean_nli_bi=wmean(g.nli_bi,w), share_equiv_0p5=wmean(g.nli_bi>=.5,w), share_equiv_0p9=wmean(g.nli_bi>=.9,w),
        share_contradiction=wmean(g.nli_con>=.5,w), FR_all=wmean(g.mean_flip,w),
        FR_equiv_0p9=wmean(g[g.nli_bi>=.9].mean_flip, g[g.nli_bi>=.9].w) if (g.nli_bi>=.9).any() else np.nan,
        n_equiv_0p9=int((g.nli_bi>=.9).sum())))
P = S.groupby("perturbation_family").apply(prof).round(4)
tot = prof(S).round(4); tot.name="ALL"; P = pd.concat([P, tot.to_frame().T]); print(P.to_string()); P.to_csv(f"{OUT_DIR}/nli_family_profile.csv")
# nli_bi vs flip (descriptive, within family)
from scipy.stats import spearmanr
r = S.groupby("perturbation_family").apply(lambda g: spearmanr(g.nli_bi, g.mean_flip)[0]).round(3); print("\nSpearman(nli_bi, mean_flip) per family:\n", r.to_string())
r.to_csv(f"{OUT_DIR}/nli_flip_spearman_by_family.csv")


# cluster bootstrap (over source reviews) of the corpus-level flip rate on NLI-equivalent pairs, threshold 0.9 and 0.5
rng = np.random.default_rng(1); keys, inv = np.unique(S.key.values, return_inverse=True); nk = len(keys)
def boot(mask, B=500):
    w = S.w.values*mask; y = S.mean_flip.values; est = (w*y).sum()/w.sum(); vals=[]
    for _ in range(B):
        cnt = np.bincount(rng.integers(0,nk,nk), minlength=nk)[inv]
        ww = w*cnt; vals.append((ww*y).sum()/ww.sum())
    return est, *np.percentile(vals,[2.5,97.5])
rows=[]
for name,mask in [("all scored pairs", np.ones(len(S),bool)), ("nli_bi >= 0.5", (S.nli_bi>=.5).values), ("nli_bi >= 0.9", (S.nli_bi>=.9).values)]:
    e,lo,hi = boot(mask); rows.append(dict(subset=name, n=int(mask.sum()), FR=round(e,4), ci_lo=round(lo,4), ci_hi=round(hi,4)))
B = pd.DataFrame(rows); print(B.to_string(index=False)); B.to_csv(f"{OUT_DIR}/nli_restricted_FR.csv", index=False)


# (c) agreement with the human audit: NLI on the 576 audited pairs, AUC vs the human meaning label; compare with Φ
from sklearn.metrics import roc_auc_score
H = pd.read_csv(HUMAN_CSV, encoding="utf-8-sig"); HR = score(H, "human")
H["nli_bi"] = np.minimum(HR[:,0], HR[:,3])
res = []
for name,y in [("fully meaning-preserving (yes vs rest)", H.meaning_preserving=="yes"), ("not meaning-altering (not 'no')", H.meaning_preserving!="no"), ("fully natural", H.natural=="yes")]:
    if y.nunique()<2: continue
    res.append(dict(target=name, AUC_nli_bi=round(roc_auc_score(y,H.nli_bi),3), AUC_phi=round(roc_auc_score(y,H.sem_preservation_score),3)))
AUC = pd.DataFrame(res); print(AUC.to_string(index=False)); AUC.to_csv(f"{OUT_DIR}/nli_vs_human_auc.csv", index=False)
print("\nmean nli_bi by human meaning label:\n", H.groupby("meaning_preserving").nli_bi.agg(["mean","count"]).round(3))
H[["sample_id","perturbation_family","meaning_preserving","nli_bi","sem_preservation_score"]].to_csv(f"{OUT_DIR}/nli_on_human_pairs.csv", index=False)
import zipfile
with zipfile.ZipFile(f"{OUT_DIR}/nli_tables.zip","w") as z:
    for f in glob.glob(f"{OUT_DIR}/*.csv"):
        if "nli_scores" not in f: z.write(f, os.path.basename(f))
print("\nsend me:", f"{OUT_DIR}/nli_tables.zip (small)")
'''
STAGES["consistency"] = r'''
FIXED_CSV = f"{BASE}/ALL_MODELS_combined_FIXED.csv"
OUT_DIR   = f"{BASE}/results_FIXED/consistency"
BACKBONE  = "distilroberta-base"
METHODS   = ["erm", "aug_prop", "cr_prop", "cr_all", "aug_all"]
SEEDS     = [0, 1, 2]
LAMBDA    = 1.0                  # weight of the consistency / augmentation term
EPOCHS, BS, LR, MAXLEN, WARMUP = 2, 32, 2e-5, 96, 0.06
SPLIT_SEED, EVAL_CAP = 0, 6000
PROP = ["P02_discourse","P08_synonym","P10_syntax","P11_entity","P12_register"]
import os; os.makedirs(OUT_DIR+"/runs", exist_ok=True)


import pandas as pd, numpy as np, os, glob, time, warnings; warnings.filterwarnings("ignore")
cols = ["source_dataset","review_idx","original_review","perturbed_review","sentiment","perturbation_family"]
df = pd.read_csv(FIXED_CSV, usecols=cols, low_memory=False)
df["key"] = df.source_dataset.astype(str)+"::"+df.review_idx.astype(str)
df["y"]   = (df.sentiment=="positive").astype(np.int64)
FAMS = sorted(df.perturbation_family.unique())
U = df.drop_duplicates("key")[["key","source_dataset","original_review","y"]].reset_index(drop=True)
assert U.y.isin([0,1]).all() and (df.groupby("key").y.nunique()==1).all()
rng = np.random.default_rng(SPLIT_SEED); perm = rng.permutation(len(U)); n=len(U)
U["split"] = "train"; U.loc[perm[:int(.20*n)],"split"] = "test"; U.loc[perm[int(.20*n):int(.25*n)],"split"] = "dev"
df["split"] = df.key.map(U.set_index("key").split)
print(len(df),"pairs |",n,"reviews |", U.split.value_counts().to_dict())
print(df.groupby("split").perturbation_family.value_counts().unstack(0).fillna(0).astype(int))


# ── fixed evaluation set (identical for every run): all test originals + capped, re-weighted test pairs ──
test_pairs = df[df.split=="test"]
sh = test_pairs.sample(frac=1, random_state=SPLIT_SEED)
EP = sh[sh.groupby("perturbation_family").cumcount() < EVAL_CAP].sort_index().reset_index(drop=True)
fam_tot = test_pairs.groupby("perturbation_family").size(); fam_n = EP.groupby("perturbation_family").size()
EP["w"] = EP.perturbation_family.map(fam_tot/fam_n).astype(float)                     # corpus-proportional weights
EO = U[U.split=="test"].reset_index(drop=True)
key2o = pd.Series(np.arange(len(EO)), index=EO.key); EP["o_idx"] = EP.key.map(key2o).values
print(len(EO),"test originals;",len(EP),"evaluation pairs (weights sum =",round(EP.w.sum()),")")

# ── training pools: for every train review, the pair indices usable by each method ──
TR = U[U.split=="train"].reset_index(drop=True)
trp = df[df.split=="train"].reset_index(drop=True)
def pools(fams):
    sub = trp[trp.perturbation_family.isin(fams)]
    return {k: v for k, v in sub.groupby("key").indices.items()}, sub.perturbed_review.values
POOLS = {"prop": pools(PROP), "all": pools(FAMS)}
print("train reviews:", len(TR), "| with a propositional pair:", len(POOLS["prop"][0]), "| with any pair:", len(POOLS["all"][0]))
METHOD_CFG = {   # name: (pool, loss on perturbed)
  "erm": (None, None), "aug_prop": ("prop","ce"), "cr_prop": ("prop","kl"), "cr_all": ("all","kl"), "aug_all": ("all","ce")}


import torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
dev = "cuda" if torch.cuda.is_available() else "cpu"; print(dev)
tok = AutoTokenizer.from_pretrained(BACKBONE)

def enc(texts): return tok(list(texts), padding=True, truncation=True, max_length=MAXLEN, return_tensors="pt").to(dev)

@torch.no_grad()
def predict(model, texts, bs=256):
    model.eval(); out = np.zeros(len(texts), dtype=np.int64); order = np.argsort([len(t) for t in texts])
    for i in range(0, len(order), bs):
        ix = order[i:i+bs]
        with torch.autocast(dev, dtype=torch.float16, enabled=(dev=="cuda")):
            out[ix] = model(**enc([texts[j] for j in ix])).logits.argmax(-1).cpu().numpy()
    return out

def sym_kl(lp, lq):   # log-prob inputs, per-example symmetric KL
    p, q = lp.exp(), lq.exp()
    return 0.5*((p*(lp-lq)).sum(-1) + (q*(lq-lp)).sum(-1))

def train(method, seed):
    torch.manual_seed(seed); rs = np.random.default_rng(seed)
    pool_name, ploss = METHOD_CFG[method]
    model = AutoModelForSequenceClassification.from_pretrained(BACKBONE, num_labels=2).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * int(np.ceil(len(TR)/BS)); sch = get_linear_schedule_with_warmup(opt, int(WARMUP*steps), steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(dev=="cuda"))
    pool, ptexts = POOLS[pool_name] if pool_name else ({}, None)
    texts, ys, keys = TR.original_review.values, TR.y.values, TR.key.values
    for ep in range(EPOCHS):
        model.train(); order = rs.permutation(len(TR)); t0 = time.time(); tot = 0.0
        for s in range(0, len(order), BS):
            ix = order[s:s+BS]; x = [texts[i] for i in ix]; y = torch.tensor(ys[ix], device=dev)
            if pool_name:   # one random usable perturbation per review (mask = review has one)
                pj = [(pool[keys[i]][rs.integers(len(pool[keys[i]]))] if keys[i] in pool else -1) for i in ix]
                m = np.array([j >= 0 for j in pj]); xp = [ptexts[j] if j >= 0 else texts[i] for i, j in zip(ix, pj)]
            with torch.autocast(dev, dtype=torch.float16, enabled=(dev=="cuda")):
                if pool_name:
                    lg = model(**enc(x + xp)).logits.float(); lo, lp = lg[:len(x)], lg[len(x):]
                    mt = torch.tensor(m, device=dev, dtype=torch.float32); ce = F.cross_entropy(lo, y)
                    if ploss == "ce":  extra = (F.cross_entropy(lp, y, reduction="none")*mt).sum()/mt.sum().clamp(min=1)
                    else:              extra = (sym_kl(F.log_softmax(lo,-1), F.log_softmax(lp,-1))*mt).sum()/mt.sum().clamp(min=1)
                    loss = ce + LAMBDA*extra
                else:
                    loss = F.cross_entropy(model(**enc(x)).logits.float(), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sch.step(); tot += loss.item()
        print(f"  {method} seed{seed} epoch {ep+1}: loss {tot/np.ceil(len(order)/BS):.4f} ({time.time()-t0:.0f}s)")
    return model


def evaluate(pred_o, pred_p):
    # pred_o: predictions on EO (test originals); pred_p: predictions on EP.perturbed_review
    corr_o = (pred_o == EO.y.values)
    po_pair = pred_o[EP.o_idx.values]                       # original prediction aligned to each pair
    flip = (po_pair != pred_p).astype(np.int8); oc = corr_o[EP.o_idx.values]
    w = EP.w.values; fam = EP.perturbation_family.values
    def wm(x, mask=None):
        mask = np.ones(len(x), bool) if mask is None else mask
        return float((x[mask]*w[mask]).sum()/w[mask].sum())
    propm = np.isin(fam, PROP)
    r = dict(acc_orig=float(corr_o.mean()), acc_pert=wm((pred_p == EP.y.values).astype(float)), FR=wm(flip), FRc=wm(flip, oc), FRw=wm(flip, ~oc),
             FR_prop=wm(flip, propm), FR_mod=wm(flip, ~propm))
    for f in FAMS: r["FR_"+f] = wm(flip, fam==f)
    r["DR"] = wm(((flip==1)&oc).astype(float)); r["IRR"] = wm(((flip==1)&~oc).astype(float))
    return r, flip

def run_one(method, seed):
    f = f"{OUT_DIR}/runs/{method}_s{seed}.npz"
    if os.path.exists(f): return
    t0 = time.time(); model = train(method, seed)
    pred_o = predict(model, EO.original_review.tolist()); pred_p = predict(model, EP.perturbed_review.tolist())
    dv = U[U.split=="dev"]; acc_dev = float((predict(model, dv.original_review.tolist()) == dv.y.values).mean())
    r, flip = evaluate(pred_o, pred_p); r.update(method=method, seed=seed, acc_dev=acc_dev, minutes=(time.time()-t0)/60)
    np.savez_compressed(f, flip=flip, pred_o=pred_o.astype(np.int8), **{k: np.array(v) for k, v in r.items()})
    print(f"[{method} s{seed}] acc_o {r['acc_orig']:.4f} acc_p {r['acc_pert']:.4f} FR {r['FR']:.4f} FR_prop {r['FR_prop']:.4f} FR_mod {r['FR_mod']:.4f}  ({r['minutes']:.1f} min)")
    del model; torch.cuda.empty_cache()


# RUN (resumable: finished runs are skipped after a disconnect). Order = seed-major so every method has seed 0 first.
for seed in SEEDS:
    for method in METHODS:
        run_one(method, seed)


# ── summary: mean ± sd over seeds, generalisation to families NOT used in training ──
R = []
for f in sorted(glob.glob(f"{OUT_DIR}/runs/*.npz")):
    z = np.load(f, allow_pickle=True); R.append({k: z[k].item() for k in z.files if k not in ("flip","pred_o")})
R = pd.DataFrame(R)
w = EP.w.values; fam = EP.perturbation_family.values
TRAINED = {"erm": [], "aug_prop": PROP, "cr_prop": PROP, "cr_all": FAMS, "aug_all": FAMS}
FL = {(r.method, int(r.seed)): np.load(f"{OUT_DIR}/runs/{r.method}_s{int(r.seed)}.npz")["flip"] for r in R.itertuples()}
def unseen_fr(row):
    m = ~np.isin(fam, TRAINED[row.method]); fl = FL[(row.method, int(row.seed))]
    return float((fl[m]*w[m]).sum()/w[m].sum()) if m.any() else np.nan
R["FR_unseen_families"] = [unseen_fr(r) for r in R.itertuples()]
show = ["acc_orig","acc_pert","FR","FRc","FRw","FR_prop","FR_mod","FR_unseen_families","DR","IRR"]
S = R.groupby("method")[show].agg(["mean","std"]).round(4).loc[[m for m in METHODS if m in R.method.unique()]]
print(S.to_string()); S.to_csv(f"{OUT_DIR}/summary_mean_sd.csv"); R.to_csv(f"{OUT_DIR}/all_runs.csv", index=False)
Fm = R.groupby("method")[[c for c in R.columns if c.startswith("FR_P")]].mean().round(4).T; print("\n", Fm.to_string()); Fm.to_csv(f"{OUT_DIR}/family_FR_by_method.csv")


# ΔFR (method − erm): average the flip indicators over seeds, then cluster-bootstrap over test reviews (same pairs for all methods)
rng = np.random.default_rng(7); keys_, inv = np.unique(EP.key.values, return_inverse=True); nk = len(keys_)
def mean_flip(method): return np.mean([FL[(method, s)] for s in SEEDS if (method, s) in FL], axis=0)
base = mean_flip("erm"); rows = []
for m in [x for x in METHODS if x != "erm" and any((x, s) in FL for s in SEEDS)]:
    d = mean_flip(m) - base
    for name, mask in [("all families", np.ones(len(d), bool)), ("propositional families", np.isin(fam, PROP)), ("modulating families", ~np.isin(fam, PROP)),
                       ("families not trained on", ~np.isin(fam, TRAINED[m]))]:
        if not mask.any(): continue
        ww = w*mask; est = (ww*d).sum()/ww.sum(); vals = []
        for _ in range(1000):
            cnt = np.bincount(rng.integers(0, nk, nk), minlength=nk)[inv]; a = ww*cnt
            vals.append((a*d).sum()/a.sum())
        lo, hi = np.percentile(vals, [2.5, 97.5]); rows.append(dict(method=m, subset=name, delta_FR=round(est,4), ci_lo=round(lo,4), ci_hi=round(hi,4), sig=bool(lo>0 or hi<0)))
D = pd.DataFrame(rows); print(D.to_string(index=False)); D.to_csv(f"{OUT_DIR}/delta_FR_vs_erm.csv", index=False)
import zipfile
with zipfile.ZipFile(f"{OUT_DIR}/consistency_tables.zip", "w") as z:
    for f in glob.glob(f"{OUT_DIR}/*.csv"): z.write(f, os.path.basename(f))
print("\nsend me:", f"{OUT_DIR}/consistency_tables.zip")
'''
STAGES["kappa"] = r'''
A1_CSV = f"{BASE}/annotated_output.csv"
A2_CSV = f"{BASE}/annotation_sheet_for_second_annotator_FILLED.csv"
OUT_DIR = f"{BASE}/results_FIXED/annotation"
import os; os.makedirs(OUT_DIR, exist_ok=True)


import pandas as pd, numpy as np
from sklearn.metrics import cohen_kappa_score, roc_auc_score, confusion_matrix
a1 = pd.read_csv(A1_CSV, encoding="utf-8-sig"); a2 = pd.read_csv(A2_CSV, encoding="utf-8-sig")
for c in ("meaning_a2","sentiment_a2","natural_a2"): a2[c] = a2[c].astype(str).str.strip().str.lower()
ORD = {"meaning":("meaning_preserving","meaning_a2",["no","partial","yes"]),
       "sentiment":("sentiment_preserving","sentiment_a2",["no","unclear","yes"]),
       "natural":("natural","natural_a2",["no","borderline","yes"])}
d = a1.merge(a2[["sample_id","meaning_a2","sentiment_a2","natural_a2"]], on="sample_id", how="inner")
print(len(a1), "rows in A1;", len(a2), "in A2;", len(d), "matched")
bad = {k:sorted(set(d[c2])-set(lv)) for k,(c1,c2,lv) in ORD.items()}
assert all(len(v)==0 for v in bad.values()), f"unexpected labels in annotator 2 file: {bad} (blank or misspelled?)"


rng = np.random.default_rng(0)
def boot_kappa(x, y, levels, weights, B=2000):
    n=len(x); ks=[]
    for _ in range(B):
        i=rng.integers(0,n,n)
        try: ks.append(cohen_kappa_score(x[i], y[i], labels=levels, weights=weights))
        except Exception: pass
    return np.nanpercentile(ks,[2.5,97.5])
rows=[]
for k,(c1,c2,lv) in ORD.items():
    x=d[c1].values; y=d[c2].values
    for w in (None,"linear"):
        kap=cohen_kappa_score(x,y,labels=lv,weights=w); lo,hi=boot_kappa(x,y,lv,w)
        rows.append(dict(dimension=k,weighting=w or "none",kappa=round(kap,3),ci_lo=round(lo,3),ci_hi=round(hi,3),agreement=round((x==y).mean(),3)))
K=pd.DataFrame(rows); print(K.to_string(index=False)); K.to_csv(f"{OUT_DIR}/kappa_overall.csv",index=False)
print("\nLandis & Koch: <0.20 slight, .21-.40 fair, .41-.60 moderate, .61-.80 substantial, >.80 almost perfect")
for k,(c1,c2,lv) in ORD.items():
    cm=pd.DataFrame(confusion_matrix(d[c1],d[c2],labels=lv),index=[f"A1_{l}" for l in lv],columns=[f"A2_{l}" for l in lv]); print("\n",k); print(cm)
    cm.to_csv(f"{OUT_DIR}/confusion_{k}.csv")


# per-family kappa on the meaning judgement (48 pairs per family -> wide uncertainty, report as descriptive)
c1,c2,lv = ORD["meaning"]; r=[]
for fam,g in d.groupby("perturbation_family"):
    x,y=g[c1].values,g[c2].values
    kap = cohen_kappa_score(x,y,labels=lv,weights="linear") if len(set(x))>1 or len(set(y))>1 else np.nan
    r.append(dict(family=fam,n=len(g),agreement=round((x==y).mean(),3),kappa_linear=None if np.isnan(kap) else round(kap,3)))
F=pd.DataFrame(r); print(F.to_string(index=False)); F.to_csv(f"{OUT_DIR}/kappa_by_family.csv",index=False)
# disagreements -> adjudicate together, record the final label in a NEW column; never overwrite A1/A2
dis = d[(d.meaning_preserving!=d.meaning_a2)|(d.sentiment_preserving!=d.sentiment_a2)|(d.natural!=d.natural_a2)]
dis[["sample_id","perturbation_family","original_review","perturbed_review","meaning_preserving","meaning_a2","sentiment_preserving","sentiment_a2","natural","natural_a2"]].assign(adjudicated_meaning="",adjudicated_sentiment="",adjudicated_natural="").to_csv(f"{OUT_DIR}/disagreements_to_adjudicate.csv",index=False)
print(len(dis),"rows with at least one disagreement ->", f"{OUT_DIR}/disagreements_to_adjudicate.csv")


# headline numbers of the paper, recomputed per annotator and for the strict consensus (both agree)
PROP=["P02_discourse","P08_synonym","P10_syntax","P11_entity","P12_register"]; MOD=["P03_negation","P04_sarcasm","P05_dilution","P07_contrastive","P09_intensity"]
def summ(lab, name):
    s=lab.copy(); out=dict(annotator=name, n=len(s))
    out["meaning_yes_partial_no"]="/".join(str(int((s.m==v).sum())) for v in ("yes","partial","no"))
    out["share_fully_preserving_propositional"]=round((s[s.perturbation_family.isin(PROP)].m=="yes").mean(),3)
    out["share_fully_preserving_modulating"]=round((s[s.perturbation_family.isin(MOD)].m=="yes").mean(),3)
    out["AUC_phi_vs_preserving"]=round(roc_auc_score(s.m=="yes", s.sem_preservation_score),3)
    out["AUC_phi_vs_not_altering"]=round(roc_auc_score(s.m!="no", s.sem_preservation_score),3) if (s.m=="no").any() else None
    return out
t=[summ(d.assign(m=d.meaning_preserving),"annotator 1"), summ(d.assign(m=d.meaning_a2),"annotator 2")]
cons=d[d.meaning_preserving==d.meaning_a2].assign(m=lambda x:x.meaning_preserving); t.append(summ(cons,"consensus (both agree)"))
T=pd.DataFrame(t); print(T.to_string(index=False)); T.to_csv(f"{OUT_DIR}/humanaudit_headlines.csv",index=False)
'''

DONE = {"nli": f"{RES}/nli/nli_tables.zip", "consistency": f"{RES}/consistency/consistency_tables.zip", "kappa": f"{RES}/annotation/humanaudit_headlines.csv"}
FLAG = {"nli": RUN_NLI, "consistency": RUN_CONSISTENCY, "kappa": RUN_KAPPA}
status = {}
for name in ["nli", "consistency", "kappa"]:
    if not FLAG[name]: status[name] = "skipped (flag off)"; continue
    if name == "kappa" and not os.path.exists(f"{BASE}/annotation_sheet_for_second_annotator_FILLED.csv"):
        status[name] = "skipped (no annotation_sheet_for_second_annotator_FILLED.csv in BASE yet - run again when annotator 2 is done)"; continue
    if os.path.exists(DONE[name]) and name != "kappa":
        status[name] = "already finished (delete its output folder to redo)"; continue
    print("\n" + "="*30, name.upper(), "="*30); t0 = time.time()
    try:
        ns = {"BASE": BASE, "__name__": "__main__"}
        exec(STAGES[name], ns)
        status[name] = f"OK ({(time.time()-t0)/60:.1f} min)"
    except Exception:
        traceback.print_exc(); status[name] = "FAILED - copy the error above and send it to me"
    finally:
        ns = None; gc.collect()
        try: torch.cuda.empty_cache()
        except Exception: pass

# bundle every small result file into ONE zip on Drive
bundle = f"{RES}/CSPF_new_results_bundle.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for sub in ("nli", "consistency", "annotation"):
        for f in glob.glob(f"{RES}/{sub}/*.csv"):
            if "nli_scores" not in os.path.basename(f): z.write(f, f"{sub}/{os.path.basename(f)}")
print("\n" + "="*60); [print(f"{k:12s}: {v}") for k, v in status.items()]
print("\nSend me this ONE file:", bundle)